In [1]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import poisson
import pymc as pm
import arviz as az
import logging
import tqdm
import os

n_cores = os.cpu_count()
logger = logging.getLogger("pymc.sampling")
logger.propagate = False

In [2]:
df_matches = pd.read_pickle("data/df_matches_2010_2024.pickle")
n_teams = df_matches["host_id"].nunique()
team_to_idx = {team: idx for idx, team in enumerate(df_matches["host_name"].unique())}

In [3]:
def prepare_df_long(df_matches: pd.DataFrame):
    df_home = df_matches.copy().rename(columns={
        "host_name": "team_name",
        "host_goals": "goals",
        "guest_name": "opponent_name",
        "guest_goals": "opponent_goals"
    })

    df_away = df_matches.copy().rename(columns={
        "host_name": "opponent_name",
        "host_goals": "opponent_goals",
        "guest_name": "team_name",
        "guest_goals": "goals"
    })

    df_home["home"] = 1.0
    df_away["home"] = 0.0

    return pd.concat([df_home, df_away], ignore_index=True)


In [4]:
class Prediction:
    def __init__(self, p_scorelines) -> None:
        self.p_scorelines = p_scorelines

    def max_util_scoreline(self):
        utilities = self._expected_utilities()
        max_index = np.unravel_index(np.argmax(utilities), utilities.shape)
        return tuple(int(x) for x in max_index)

    def plot_joint_pred(self):
        plt.figure()
        sns.heatmap(self.p_scorelines, annot=True, fmt=".2f", cmap="YlGnBu")
        plt.title("Joint Probability Distribution of Predicted Scorelines")
        plt.xlabel("Away Team Goals")
        plt.ylabel("Home Team Goals")
        plt.show()
    
    def plot_exp_utilities(self):
        utilities = self._expected_utilities()
        plt.figure()
        sns.heatmap(utilities, annot=True, fmt=".2f", cmap="YlGnBu")
        plt.title("Expected Utilities")
        plt.xlabel("Away Team Goals")
        plt.ylabel("Home Team Goals")
        plt.show()
    
    @staticmethod
    def plot_score_rule(correct_result, max_goals=4):
        x = np.arange(0, max_goals + 1)
        y = [Prediction._scoring_rule((i, j), correct_result) for i in x for j in x]
        sns.heatmap(np.array(y).reshape(max_goals + 1, max_goals + 1), annot=True, fmt=".2f", cmap="YlGnBu")
        plt.title("Scoring Rule Heatmap")
        plt.xlabel("Predicted Away Goals")
        plt.ylabel("Predicted Home Goals")
        plt.show()

    def _expected_utilities(self):
        utilities = np.zeros_like(self.p_scorelines)
        for i in range(self.p_scorelines.shape[0]):
            for j in range(self.p_scorelines.shape[1]):
                utilities[i, j] = self._expected_utility((i, j))
        return utilities
    
    def _expected_utility(self, ground_truth):
        ep = 0
        for i in range(self.p_scorelines.shape[0]):
            for j in range(self.p_scorelines.shape[1]):
                ep += Prediction._scoring_rule((i, j), ground_truth) * self.p_scorelines[i, j]
        return ep

    @staticmethod
    def _scoring_rule(pred, ground_truth):
        home_pred, away_pred = pred
        home_gt, away_gt = ground_truth

        if home_pred == home_gt and away_pred == away_gt:
            return 4 # correct result
        if home_pred - away_pred == home_gt - away_gt:
            return 2 if home_pred == away_pred else 3 # correct difference
        if home_pred > away_pred and home_gt > away_gt or home_pred < away_pred and home_gt < away_gt:
            return 2 # correct trend
        return 0 # incorrect

assert Prediction._scoring_rule((1, 0), (1, 0)) == 4 # correct
assert Prediction._scoring_rule((1, 0), (2, 0)) == 2 # correct trend
assert Prediction._scoring_rule((1, 0), (0, 0)) == 0 # incorrect
assert Prediction._scoring_rule((2, 2), (1, 1)) == 2 # correct difference (draw)
assert Prediction._scoring_rule((3, 2), (2, 1)) == 3 # correct difference (no draw)

class PoissonModel:
    def fit(self, df_long):
        self.model = smf.glm(
            formula="goals ~ home + team_name + opponent_name",
            data=df_long,
            family=sm.families.Poisson()
        ).fit()
    
    def predict(self, home_name, guest_name, max_goals=4):                
        new_match = pd.DataFrame({
            'team_name': [home_name, guest_name],
            'opponent_name': [guest_name, home_name],
            'home': [1, 0]
        })
        try:
            home_exp, away_exp = self.model.predict(new_match)
        except Exception as e:
            print(f"Error predicting match: {e}")
            home_exp, away_exp = 0, 0

        home_goals_prob = [poisson.pmf(i, home_exp) for i in range(max_goals)]
        away_goals_prob = [poisson.pmf(i, away_exp) for i in range(max_goals)]

        # sum tails
        home_goals_prob.append(1 - np.sum(home_goals_prob))
        away_goals_prob.append(1 - np.sum(away_goals_prob))

        p_scorelines = np.outer(home_goals_prob, away_goals_prob)

        return Prediction(p_scorelines)

In [5]:
def to_probs(samples: np.ndarray, max_goals: int):
    return np.bincount(
        np.clip(samples, 0, max_goals), minlength=max_goals + 1
    ) / len(samples)


class BayesianPoissonModel:

    def fit(self, df_long, draws=2000, tune=3000, target_accept=0.95):
        with pm.Model() as model:
            # Priors
            intercept = pm.Normal("intercept", mu=0, sigma=5)
            home_adv = pm.Normal("home_adv", mu=0, sigma=5)

            # Team attack/defense priors
            team_attack = pm.Normal("team_attack", mu=0, sigma=3, shape=n_teams)
            team_defense = pm.Normal("team_defense", mu=0, sigma=3, shape=n_teams)

            # Home/Away effects
            home = pm.Data("home", df_long['home'].values)
            team_index = pm.Data("team_index", df_long['team_name'].map(team_to_idx))
            opponent_index = pm.Data("opponent_index", df_long['opponent_name'].map(team_to_idx))

            # Expected goals
            mu = pm.math.exp(
                intercept
                + home_adv * home
                + team_attack[team_index]
                - team_defense[opponent_index]
            )

            # Likelihood
            pm.Poisson("goals_obs", mu=mu, observed=df_long['goals'].values, shape=team_index.shape[0])

            # Sample posterior
            self.idata = pm.sample(draws=draws, tune=tune, target_accept=target_accept, cores=n_cores, chains=10)
            self.model = model
        
    def predict_batch(self, df_matches, max_goals=4):
        df_long = prepare_df_long(df_matches)
        with self.model:
            pm.set_data({
                "home": df_long['home'].values,
                "team_index": df_long['team_name'].map(team_to_idx).values,
                "opponent_index": df_long['opponent_name'].map(team_to_idx).values
            })
            post_idata = pm.sample_posterior_predictive(
                self.idata, var_names=["goals_obs"], progressbar=False, return_inferencedata=True
            )
        values = post_idata.posterior_predictive['goals_obs'].stack(sample=("chain", "draw")).values
        probs = np.apply_along_axis(to_probs, 1, values, max_goals=max_goals)

        probs_home = probs[:len(probs) // 2]
        probs_guest = probs[len(probs) // 2:]

        p_scorelines = np.einsum('ij,ik->ijk', probs_home, probs_guest)
        return [Prediction(p_scorelines[i]) for i in range(len(p_scorelines))]
        
    
    def predict(self, home_name, guest_name, max_goals=4):
        with self.model:
            pm.set_data({
                "home": [1, 0],
                "team_index": [team_to_idx[home_name], team_to_idx[guest_name]],
                "opponent_index": [team_to_idx[guest_name], team_to_idx[home_name]]
            })
            post_idata = pm.sample_posterior_predictive(
                self.idata, var_names=["goals_obs"], progressbar=False, return_inferencedata=True
            )

        values = post_idata.posterior_predictive['goals_obs'].stack(sample=("chain", "draw"))

        home_goals = values.sel(goals_obs_dim_0=0).values
        home_goals_probs = to_probs(home_goals, max_goals=4)

        guest_goals = values.sel(goals_obs_dim_0=1).values
        guest_goals_probs = to_probs(guest_goals, max_goals=max_goals)

        p_scorelines = np.outer(home_goals_probs, guest_goals_probs)
        return Prediction(p_scorelines)


In [6]:
df_matches_all = pd.read_pickle("data/df_matches_2010_2024.pickle")
total_scores = []
for test_season in range(2012, 2025):
    season_diff = test_season - df_matches_all["season"]
    df_matches = df_matches_all[(season_diff < 3) & (season_diff > 0)]
    df_long = prepare_df_long(df_matches)
    model = PoissonModel()
    model.fit(df_long)

    df_matches_test = df_matches_all[(df_matches_all["season"] == test_season) & (df_matches_all["league"] == "bl1")]
    score = 0
    for _, row in df_matches_test.iterrows():
        home_team = row["host_name"]
        away_team = row["guest_name"]
        correct_result = (row["host_goals"], row["guest_goals"])
        pred = model.predict(home_team, away_team)
        score += Prediction._scoring_rule(pred.max_util_scoreline(), correct_result)

    print(f"Test season {test_season}: {score}")
    total_scores.append(score)

print("Mean score:", np.mean(total_scores))
print("Std deviation:", np.std(total_scores))

Test season 2012: 402
Test season 2013: 405
Test season 2014: 406
Test season 2015: 403
Test season 2016: 375
Test season 2017: 367
Test season 2018: 378
Test season 2019: 405
Test season 2020: 385
Test season 2021: 373
Test season 2022: 426
Test season 2023: 382
Test season 2024: 382
Mean score: 391.46153846153845
Std deviation: 16.703824456671292


In [ ]:
df_matches_all = pd.read_pickle("data/df_matches_2010_2024.pickle")
total_scores = []
for test_season in range(2012, 2025):
    season_diff = test_season - df_matches_all["season"]
    df_matches = df_matches_all[(season_diff < 3) & (season_diff > 0)]
    df_long = prepare_df_long(df_matches)
    model = BayesianPoissonModel()
    model.fit(df_long)

    df_matches_test = df_matches_all[(df_matches_all["season"] == test_season) & (df_matches_all["league"] == "bl1")]
    predictions = model.predict_batch(df_matches_test)

    score = 0
    for i in tqdm.tqdm(range(len(df_matches_test))):
        correct_result = (df_matches_test.iloc[i]["host_goals"], df_matches_test.iloc[i]["guest_goals"])
        prediction = predictions[i]
        score += Prediction._scoring_rule(prediction.max_util_scoreline(), correct_result)

    print(f"Test season {test_season}: {score}")
    total_scores.append(score)

print("Mean score:", np.mean(total_scores))
print("Std deviation:", np.std(total_scores))

Output()

The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details
100%|██████████| 306/306 [00:00<00:00, 1437.07it/s]


Test season 2012: 400


Output()

The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details
100%|██████████| 306/306 [00:00<00:00, 1279.99it/s]


...

Test season 2014: 398


Output()

The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details
100%|██████████| 306/306 [00:00<00:00, 1445.69it/s]


Test season 2015: 400


Output()